# 01_BIS_exploration_francetravail_SCRIPTS.ipynb

In [9]:
import requests
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os

load_dotenv()

CLIENT_ID     = os.getenv("FRANCETRAVAIL_CLIENT_ID")
CLIENT_SECRET = os.getenv("FRANCETRAVAIL_CLIENT_SECRET")

In [10]:
TOKEN_URL = "https://entreprise.francetravail.fr/connexion/oauth2/access_token"
BASE_URL  = "https://api.francetravail.io/partenaire/offresdemploi/v2/offres"

In [11]:
class FranceTravailClient:
    """
    Client HTTP pour l'API FranceTravail.
    Gère l'authentification OAuth2 et les appels à l'API.
    """
    def __init__(self):
        self._token = None
        self._token_expiry = None

    def _get_token(self) -> str:
        """
        Récupère un token OAuth2 valide.
        Si le token existant est encore valide, le réutilise.
        Sinon, en demande un nouveau.
        """
        # Réutiliser le token si encore valide
        if self._token and datetime.now() < self._token_expiry:
            return self._token
    
        response = requests.post(
            TOKEN_URL,
            params={"realm": "/partenaire"},
            data={
                "grant_type":    "client_credentials",
                "client_id":     CLIENT_ID,
                "client_secret": CLIENT_SECRET,
                "scope":         "api_offresdemploiv2 o2dsoffre",
            }
        )
        response.raise_for_status()
        data = response.json()
    
        self._token = data["access_token"]
        # Le token expire dans expires_in secondes — on retire 30s de marge
        self._token_expiry = datetime.now() + timedelta(
            seconds=data["expires_in"] - 30
        )
        return self._token


    def _headers(self) -> dict:
        """Retourne les headers HTTP avec le token valide."""
        return {
            "Authorization": f"Bearer {self._get_token()}",
            "Accept":        "application/json",
        }

    def rechercher_offres(self, params: dict) -> dict:
        """
        Appel à l'endpoint de recherche d'offres.
        params : dictionnaire des paramètres de recherche
        Retourne le JSON brut de la réponse.
        """
        response = requests.get(
            f"{BASE_URL}/search",
            headers=self._headers(),
            params=params,
        )
        response.raise_for_status()
        return response.json()

    def get_offre(self, offre_id: str) -> dict:
        """
        Récupère le détail complet d'une offre par son identifiant.
        """
        response = requests.get(
            f"{BASE_URL}/{offre_id}",
            headers=self._headers(),
        )
        response.raise_for_status()
        return response.json()

In [12]:
# ingestion/francetravail/extractor.py

import json
import time
from datetime import datetime
from pathlib import Path
# from ingestion.francetravail.api_client import FranceTravailClient

# Nombre maximum d'offres par page autorisé par l'API
PAGE_SIZE = 100

def extraire_offres(
    mots_cles:    str = "data engineer",
    nb_pages_max: int = 10,
    avec_details: bool = False,
) -> list:
    """
    Extrait les offres FranceTravail pour un mot-clé donné.

    mots_cles    : termes de recherche
    nb_pages_max : nombre de pages à parcourir (100 offres par page)
    avec_details : si True, récupère le détail complet de chaque offre

    Retourne une liste de dictionnaires bruts.
    """
    client  = FranceTravailClient()
    offres  = []
    page    = 0

    while page < nb_pages_max:
        debut = page * PAGE_SIZE
        fin   = debut + PAGE_SIZE - 1

        params = {
            "motsCles": mots_cles,
            "range":    f"{debut}-{fin}",
            "sort":     "1",  # tri par date de publication
        }

        print(f"Page {page + 1} — offres {debut} à {fin}...")

        try:
            data = client.rechercher_offres(params)
        except Exception as e:
            print(f"Erreur page {page + 1} : {e}")
            break

        resultats = data.get("resultats", [])

        if not resultats:
            print("Plus d'offres disponibles, arrêt.")
            break

        # Optionnel : enrichir avec le détail complet de chaque offre
        '''
        if avec_details:
            resultats = _enrichir_avec_details(client, resultats)
        '''

        offres.extend(resultats)
        page += 1

        # Délai poli entre les pages
        time.sleep(0.5)

    print(f"\nTotal extrait : {len(offres)} offres")
    return offres


def _enrichir_avec_details(
    client:    FranceTravailClient,
    resultats: list,
) -> list:
    """
    Pour chaque offre de la liste, récupère son détail complet.
    Utile pour avoir la description complète et toutes les compétences.
    """
    enrichis = []
    for offre in resultats:
        offre_id = offre.get("id")
        if not offre_id:
            enrichis.append(offre)
            continue
        try:
            detail = client.get_offre(offre_id)
            enrichis.append(detail)
            time.sleep(0.2)  # respecter le rate limit
        except Exception as e:
            print(f"Impossible de récupérer le détail {offre_id} : {e}")
            enrichis.append(offre)  # on garde la version partielle
    return enrichis


def sauvegarder_brut(offres: list, mots_cles: str = "") -> str:
    """
    Sauvegarde les données brutes dans data/raw/francetravail/
    avec un timestamp dans le nom de fichier.
    Retourne le chemin du fichier créé.
    """
    Path("../data/raw/francetravail").mkdir(parents=True, exist_ok=True)

    timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
    slug        = mots_cles.replace(" ", "_") if mots_cles else "offres"
    chemin      = f"../data/raw/francetravail/{slug}_{timestamp}.json"

    with open(chemin, "w", encoding="utf-8") as f:
        json.dump(offres, f, ensure_ascii=False, indent=2)

    print(f"Sauvegardé : {chemin}")
    return chemin


if __name__ == "__main__":
    # Point d'entrée — lance l'extraction directement
    mots_cles = "data engineer"

    offres = extraire_offres(
        mots_cles    = mots_cles,
        nb_pages_max = 5,
        avec_details = False,  # passer à True pour les descriptions complètes
    )

    sauvegarder_brut(offres, mots_cles)

Page 1 — offres 0 à 99...
Page 2 — offres 100 à 199...
Page 3 — offres 200 à 299...
Page 4 — offres 300 à 399...
Page 5 — offres 400 à 499...

Total extrait : 500 offres
Sauvegardé : ../data/raw/francetravail/data_engineer_20260331_145054.json
